# Create ANII Uruguay Awards (GRANT PATTERN, official project portal)

ANII (Agencia Nacional de Investigacion e Innovacion, Uruguay) publishes an official project portal at `https://anii.org.uy/proyectos/`. The rendered page loads its table from ANII's own AJAX endpoint (`/app/ajax/frontend/getMoreProjects.php`), and each project detail page publishes the native code, instrument, beneficiary, subsidy amount/currency, start date, duration, call year, status, and public summary.

**Awarding body:** ANII Uruguay -- F4320310753 (ROR 03egaj678, DOI 10.13039/100008725)

**Scope choices:**
- One row per ANII research project.
- Include only portal rows whose broad area is `INVESTIGACION`; exclude `FORMACION`, `INNOVACION`, and `EMPRENDIMIENTOS` because those categories include scholarships, mobility/training, startups, and non-research support.
- `funder_award_id` = `anii-` + ANII native project code.
- `funder_scheme` = ANII instrument (for example, Innovagro) from the visible detail-page field.
- `amount` and `currency` are parsed from the visible `SUBSIDIO` field instead of hardcoded. Expect UYU for most/all records, but this notebook keeps the source currency value.
- `start_date` comes from `FECHA DE INICIO`; `end_date` is derived from start date + source `DURACION` months - 1 day.
- `lead_investigator` is parsed from ANII's usual `Person : Institution` beneficiary pattern. If no person is present, the lead is NULL rather than inventing one.

**Prerequisites:** Run `scripts/local/anii_to_s3.py` first and upload the parquet to S3. Contractor local validation used `--skip-upload`; admin must run the upload and notebook in Databricks.

**Data source:** https://anii.org.uy/proyectos/
**S3 location:** `s3a://openalex-ingest/awards/anii/anii_projects.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.anii_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/anii/anii_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) AS anii_raw_rows FROM openalex.awards.anii_raw;

## Step 1.5: Inspect raw + money/currency scan

Per the runbook, inspect the raw schema and explicitly verify money/currency fields. The source script parses the visible `SUBSIDIO` field into `amount` and `currency`, but we still inspect the distribution here to catch markup or unit drift.

Known raw columns are all STRING per runbook: `funder_award_id`, `project_code`, `title`, `description`, `instrument`, `beneficiary`, `beneficiary_person`, `beneficiary_institution`, `subsidy_raw`, `amount`, `currency`, `start_date`, `duration_months`, `call_year`, `phase`, `status`, `source_url`, and table metadata.

In [ ]:
%sql
DESCRIBE openalex.awards.anii_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.anii_raw LIMIT 5;

In [ ]:
%sql
-- Money-shape scan per runbook Step 1.5 / 6.7.
SELECT
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    AVG(TRY_CAST(amount AS DOUBLE)) AS avg_amount,
    COUNT(amount) AS rows_with_amount,
    COUNT(*) AS total_rows
FROM openalex.awards.anii_raw;

In [ ]:
%sql
SELECT currency, COUNT(*) AS rows
FROM openalex.awards.anii_raw
GROUP BY currency
ORDER BY rows DESC;

## Step 1.6: Fail-fast -- verify ANII funder row exists

The transform cross joins against `openalex.common.funder`. If this returns zero rows, STOP; the insert would silently emit no awards.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320310753;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.anii_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320310753  -- ANII Uruguay
), typed AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS amount_double,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS start_date_typed,
        TRY_CAST(duration_months AS INT) AS duration_months_int
    FROM openalex.awards.anii_raw
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(a.funder_award_id)
    ))) % 9000000000 AS id,
    a.title AS display_name,
    a.description,
    f.funder_id,
    a.funder_award_id,
    a.amount_double AS amount,
    CASE WHEN a.amount_double IS NOT NULL THEN a.currency ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'research' AS funding_type,
    a.instrument AS funder_scheme,
    'anii_projects_portal' AS provenance,
    a.start_date_typed AS start_date,
    CASE
        WHEN a.start_date_typed IS NOT NULL AND a.duration_months_int IS NOT NULL
        THEN date_sub(add_months(a.start_date_typed, a.duration_months_int), 1)
        ELSE NULL
    END AS end_date,
    YEAR(a.start_date_typed) AS start_year,
    CASE
        WHEN a.start_date_typed IS NOT NULL AND a.duration_months_int IS NOT NULL
        THEN YEAR(date_sub(add_months(a.start_date_typed, a.duration_months_int), 1))
        ELSE NULL
    END AS end_year,
    CASE
        WHEN a.beneficiary_person IS NULL OR a.beneficiary_person = '' THEN NULL
        ELSE struct(
            a.lead_given_name AS given_name,
            a.lead_family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            CASE
                WHEN a.beneficiary_institution IS NULL OR a.beneficiary_institution = '' THEN CAST(NULL AS STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>)
                ELSE struct(
                    a.beneficiary_institution AS name,
                    'UY' AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                )
            END AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    a.source_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(a.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM typed a
CROSS JOIN funder_resolved f
WHERE a.funder_award_id IS NOT NULL
  AND a.title IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 82

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'anii_projects_portal' AND priority = 82;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    82 AS priority  -- ANII priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.anii_awards;

## Step 6: Verification

Full local verification must be run by an admin in Databricks after upload. Amount/currency coverage is not waived because ANII publishes `SUBSIDIO` on detail pages.

In [ ]:
%sql
SELECT COUNT(*) AS total_anii_award_rows FROM openalex.awards.anii_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.anii_awards;

In [ ]:
%sql
-- Data completeness (runbook canonical shape)
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_dates,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description
FROM openalex.awards.anii_awards;

In [ ]:
%sql
-- Step 6.7 amount + currency fail-fast check (NOT waived for ANII).
-- Expect high pct_amount and source currencies from the visible SUBSIDIO field
-- (local full run saw mostly UYU plus USD/EUR cofunded/collaboration rows).
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount
FROM openalex.awards.anii_awards;

In [ ]:
%sql
-- Native-code uniqueness check; must return zero rows.
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.anii_awards
GROUP BY funder_award_id
HAVING COUNT(*) > 1;

In [ ]:
%sql
-- Instrument distribution.
SELECT funder_scheme, COUNT(*) AS rows, ROUND(SUM(amount) / 1000000, 2) AS total_millions
FROM openalex.awards.anii_awards
GROUP BY funder_scheme
ORDER BY rows DESC;

In [ ]:
%sql
-- Year distribution.
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.anii_awards
GROUP BY start_year
ORDER BY start_year DESC;

In [ ]:
%sql
-- Sample inspection.
SELECT
    funder_award_id,
    SUBSTRING(display_name, 1, 80) AS title,
    funder_scheme,
    amount,
    currency,
    start_date,
    end_date,
    lead_investigator.given_name AS given_name,
    lead_investigator.family_name AS family_name,
    lead_investigator.affiliation.name AS affiliation
FROM openalex.awards.anii_awards
ORDER BY start_year DESC NULLS LAST, amount DESC NULLS LAST
LIMIT 20;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.anii_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;